# 딥러닝실습 중간고사

**목표: defalut_2 AlexNet best 64.1% 넘기기**  

참고 논문:  
- Wen et al., *Distract Your Attention: Multi-Head Cross Attention Network for FER*, Biomimetics 2023  
- Tan & Le, *EfficientNet: Rethinking Model Scaling for CNNs*, ICML 2019

# Part 1. 셋업

## 1-1. 라이브러리 설치

In [ ]:
import os
import random
import glob
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib
import seaborn as sns

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score

import numpy as np


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights


from facenet_pytorch import MTCNN

# 한글 폰트
plt.rcParams['font.family'] = 'malgun gothic'
plt.rcParams['axes.unicode_minus'] = False

print('torch:', torch.__version__)
print('CUDA :', torch.cuda.is_available())

## 1-2. 시드 고정 + device + 경로

In [11]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 경로
ROOT_PATH    = r'C:\Users\user\Documents\50.2026\53.DL_실습\mid_test_image\raw_data'
IMAGE_FOLDER = os.path.join(ROOT_PATH, 'Video', 'Images')
LABEL_PATH   = os.path.join(ROOT_PATH, 'mosi_audio_metadata.csv')

print(f'Seed fixed: {SEED}')
print(f'device: {device}')
print(f'IMAGE_FOLDER: {IMAGE_FOLDER}')

Seed fixed: 42
device: cuda
IMAGE_FOLDER: C:\Users\user\Documents\50.2026\53.DL_실습\mid_test_image\raw_data\Video\Images


## 1-3. 라벨 로드

In [12]:
label_df = pd.read_csv(LABEL_PATH)

# 긍정:1, 부정:0
label_dict = {}
for _, row in label_df.iterrows():
    key = os.path.splitext(row['file_name'])[0]
    label_dict[key] = row['sentiment']

print(f'라벨 수: {len(label_dict)}')

라벨 수: 2199


In [ ]:
print(label_df.columns.tolist())
print(label_df.head())
print('\nsentiment 값 분포:')
print(label_df['sentiment'].value_counts())
print('\nsentiment dtype:', label_df['sentiment'].dtype)


# Part 2. 데이터 전처리

## 2-1. face crop + resize
face crop을 찾아보니 label 데이터 없이도 크롭이 가능했다. 실습시간에서 배운 Haar Cascade 방식보다 더 안정적이고 효과적인 전처리 방식인 MTCNN을 사용해보기로 했다. margin이 있어서 padding은 생략

In [13]:
mtcnn = MTCNN(
    image_size=224, #크기
    margin=20, #마진값
    keep_all=False, # 전체 보관x 크롭된 얼굴 하나만
    device=device # gpu 
)


def face_crop_mtcnn(img):
    boxes, probs = mtcnn.detect(img) #얼굴좌표 + 얼굴일확률

    if boxes is not None:
        box = boxes[0].astype(int)
        face_img = img.crop(box)
        return face_img, True
        
    return img, False


print('mtcnn 완료')

mtcnn 완료


In [14]:
def resize_img(img, size=(224, 224)): #혹시나....
    return cv2.resize(img, size)

## 2-2. 품질 향상   
노이즈 제거, 샤프닝 효과

In [15]:
def enhance_face(pil_img):
    img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

    # 노이즈 제거 
    img = cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)

    # 샤프닝 (언샤프 마스크 필터)
    gaussian = cv2.GaussianBlur(img, (0, 0), 2.0)
    img = cv2.addWeighted(img, 1.5, gaussian, -0.5, 0)
    
    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

## 2-3. 데이터셋 만들기

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

# train용 transform (augmentation + 정규화)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# test용 transform (정규화만)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

class FaceDataset(Dataset):
    def __init__(self, fnames, labels, img_dir, transform=None):
        self.fnames    = fnames
        self.labels    = labels
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.fnames)

    def __getitem__(self, idx):
        fname = self.fnames[idx]
        label = self.labels[idx]

        # 전체 경로면 그대로, 파일명만이면 img_dir 붙이기
        if os.path.isabs(fname):
            img_path = fname
        else:
            img_path = os.path.join(self.img_dir, fname)

        # 1. 이미지 로드 (PIL)
        img = Image.open(img_path).convert('RGB')

        # 2. face crop (MTCNN, PIL 필수)
        img, found = face_crop_mtcnn(img)
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)

        # 3. 224x224 resize
        img = img.resize((224, 224))

        # 4. 품질 향상 (노이즈 제거 + 샤프닝)
        img = enhance_face(img)

        # 5. transform (augmentation + ToTensor + Normalize)
        if self.transform:
            img = self.transform(img)

        return img, label


In [ ]:
# 경로 및 파일 존재 확인
print(f'IMAGE_FOLDER 존재: {os.path.exists(IMAGE_FOLDER)}')
if os.path.exists(IMAGE_FOLDER):
    sample = os.listdir(IMAGE_FOLDER)[:5]
    print(f'폴더 내 샘플: {sample}')

# jpg / png / jpeg 모두 탐색
all_files = (
    glob.glob(os.path.join(IMAGE_FOLDER, '**', '*.jpg'), recursive=True) +
    glob.glob(os.path.join(IMAGE_FOLDER, '**', '*.jpeg'), recursive=True) +
    glob.glob(os.path.join(IMAGE_FOLDER, '**', '*.png'), recursive=True)
)

print(f'glob 탐색 파일 수: {len(all_files)}')
if len(all_files) > 0:
    print(f'샘플 파일명: {os.path.basename(all_files[0])}')

fnames_all = []
labels_all = []

for fpath in all_files:
    fname     = os.path.basename(fpath)
    fname_key = os.path.splitext(fname)[0]
    if fname_key in label_dict:
        fnames_all.append(fpath)          # 전체 경로 저장 (하위폴더 대비)
        labels_all.append(1 if label_dict[fname_key] >= 0 else 0)

print(f'라벨 매칭 파일 수: {len(fnames_all)}, 긍정: {sum(labels_all)}, 부정: {len(labels_all)-sum(labels_all)}')

# train / test split (8:2)
fnames_train, fnames_test, labels_train, labels_test = train_test_split(
    fnames_all, labels_all, test_size=0.2, random_state=SEED, stratify=labels_all
)

print(f'train: {len(fnames_train)}, test: {len(fnames_test)}')

# WeightedRandomSampler - 클래스 불균형 보정
class_counts = [labels_train.count(0), labels_train.count(1)]
weights      = [1.0 / class_counts[l] for l in labels_train]
sampler      = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_dataset = FaceDataset(fnames_train, labels_train, IMAGE_FOLDER, transform=train_transform)
test_dataset  = FaceDataset(fnames_test,  labels_test,  IMAGE_FOLDER, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print('DataLoader 준비 완료')


---
# Part 3. 모델 정의

## 3-1. EfficientNet-B0 fine-tuning

- defalut_2의 AlexNet fine-tuning과 동일한 방식
- pretrained 가중치 로드 → 마지막 FC 레이어만 교체 → 전체 학습
- EfficientNet-B0은 AlexNet(57M)보다 파라미터가 적으면서 성능은 더 높음


In [ ]:
# pretrained 가중치 로드
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

# 마지막 FC 레이어 교체 (2클래스 분류)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 2)

model = model.to(device)

# 파라미터 수 확인
total_params = sum(p.numel() for p in model.parameters())
print(f'EfficientNet-B0 파라미터 수: {total_params:,}')


---
# Part 4. 학습

## 4-1. train_model 함수

In [ ]:
def train_model(model, train_loader, criterion, optimizer, num_epochs, device):
    train_loss_list = []
    train_acc_list  = []

    for epoch in range(num_epochs):
        model.train()

        running_loss = 0.0
        correct      = 0
        total        = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted  = torch.max(outputs, 1)
            correct      += (predicted == labels).sum().item()
            total        += labels.size(0)

        epoch_loss = running_loss / total
        epoch_acc  = correct / total

        train_loss_list.append(epoch_loss)
        train_acc_list.append(epoch_acc)

        if (epoch + 1) % 5 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}]  Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}')

    return train_loss_list, train_acc_list


## 4-2. evaluate_model 함수

In [ ]:
def evaluate_model(model, data_loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct      = 0
    total        = 0
    all_preds    = []
    all_labels   = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted  = torch.max(outputs, 1)
            correct      += (predicted == labels).sum().item()
            total        += labels.size(0)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc  = correct / total

    return epoch_loss, epoch_acc, all_preds, all_labels


## 4-3. 학습 실행

In [ ]:
criterion  = nn.CrossEntropyLoss()
optimizer  = torch.optim.Adam(model.parameters(), lr=1e-4)
num_epochs = 20

print('=== 학습 시작 ===')
train_loss_list, train_acc_list = train_model(
    model, train_loader, criterion, optimizer, num_epochs, device
)

print('\n=== 테스트 평가 ===')
test_loss, test_acc, test_preds, test_labels = evaluate_model(
    model, test_loader, criterion, device
)
print(f'Test Loss: {test_loss:.4f}  Test Acc: {test_acc:.4f}')


---
# Part 5. 시각화

## Viz 1 — 모델별 파라미터 수 비교

In [ ]:
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# defalut_2 모델 파라미터 수 (AlexNet=57.01M, SimpleCNN은 작음)
param_names  = ['SimpleCNN\n(MaxPool)', 'SimpleCNN\n(AvgPool)', 'AlexNet\n(defalut_2)', 'EfficientNet-B0\n(my model)']
param_counts = [0.057, 0.057, 57.01, total_params / 1e6]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(param_names, param_counts, color=['steelblue', 'steelblue', 'steelblue', 'tomato'])
ax.set_ylabel('파라미터 수 (M)')
ax.set_title('모델별 파라미터 수 비교')
for bar, val in zip(bars, param_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.2f}M', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('viz_01_params.png', dpi=150)
plt.show()
print('viz_01_params.png 저장 완료')


## Viz 2 — 학습 곡선 (Loss & Accuracy)

In [ ]:
ep_axis = range(1, num_epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ep_axis, train_loss_list, marker='o', color='tomato', label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('학습 Loss 곡선')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(ep_axis, train_acc_list, marker='o', color='steelblue', label='Train Acc')
axes[1].axhline(y=test_acc, color='tomato', linestyle='--', label=f'Test Acc ({test_acc:.3f})')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('학습 Accuracy 곡선')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('viz_02_learning_curve.png', dpi=150)
plt.show()
print('viz_02_learning_curve.png 저장 완료')


## Viz 3 — Test Accuracy 비교 (defalut_2 vs my model)

In [ ]:
# defalut_2 결과값 (노트북 실행 후 직접 기록)
result_names = ['SimpleCNN\n(MaxPool)', 'SimpleCNN\n(AvgPool)', 'AlexNet\n(defalut_2)', 'EfficientNet-B0\n(my model)']
result_accs  = [0.600, 0.582, 0.641, test_acc]

colors = ['steelblue', 'steelblue', 'steelblue', 'tomato']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(result_names, result_accs, color=colors)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Test Accuracy')
ax.set_title('모델별 Test Accuracy 비교')
for bar, val in zip(bars, result_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('viz_03_accuracy_compare.png', dpi=150)
plt.show()
print('viz_03_accuracy_compare.png 저장 완료')


## Viz 4 — Confusion Matrix

In [ ]:
cm = confusion_matrix(test_labels, test_preds)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)

classes = ['Negative (0)', 'Positive (1)']
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(classes)
ax.set_yticklabels(classes)
ax.set_xlabel('예측 레이블')
ax.set_ylabel('실제 레이블')
ax.set_title('Confusion Matrix')

total = cm.sum()
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i,j]}\n({cm[i,j]/total*100:.1f}%)',
                ha='center', va='center', fontsize=11,
                color='white' if cm[i,j] > cm.max()/2 else 'black')

plt.tight_layout()
plt.savefig('viz_04_confusion_matrix.png', dpi=150)
plt.show()

print(classification_report(test_labels, test_preds, target_names=['Negative', 'Positive']))
print('viz_04_confusion_matrix.png 저장 완료')
